# AirScript (Google Colab Version)
Since Google Colab runs on a remote server, it cannot directly access your local webcam using standard OpenCV `cv2.VideoCapture(0)`. We use Javascript to capture frames from your browser, send them to Python for processing, and then draw the output back.

**Note:** Due to the complex asynchronous nature of sending frames back and forth to the browser, this Colab version is simplified to show core hand tracking.

In [ ]:
!pip install mediapipe opencv-python-headless

In [ ]:
import cv2
import mediapipe as mp
import numpy as np
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
import base64

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(min_detection_confidence=0.7, min_tracking_confidence=0.5)
mp_draw = mp.solutions.drawing_utils

In [ ]:
# JavaScript to capture a frame from the webcam
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = base64.b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

In [ ]:
try:
  filename = take_photo()
  print('Saved to {}'.format(filename))
  
  # Show the image which was just taken.
  img = cv2.imread(filename)
  img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
  
  results = hands.process(img_rgb)
  if results.multi_hand_landmarks:
      for hand_lms in results.multi_hand_landmarks:
          mp_draw.draw_landmarks(img, hand_lms, mp_hands.HAND_CONNECTIONS)
          
  cv2_imshow(img)
except Exception as err:
  # Errors will be thrown if the user does not have a webcam or if they do not
  # grant the page permission to access it.
  print(str(err))